In [2]:
#start session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("wheat_futures_price_prediction").getOrCreate()

25/04/21 13:24:09 WARN Utils: Your hostname, Presleys-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.1.134.32 instead (on interface en0)
25/04/21 13:24:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/21 13:24:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


25/04/21 13:24:24 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
## Importing Required Libraries
from pyspark.sql.functions import col, isnan, when, count
from pyspark.sql.functions import to_date, col
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt



#Read weather and pricing data
weather_raw = spark.read.csv('weather_data_RAW.csv', header=True, inferSchema=True)
wheat_price_data = spark.read.csv('US_wheat_future_pricing_RAW.csv', header=True, inferSchema=True)

#Parse date columns
weather = weather_raw.withColumn(
    "Date",
    to_date(col("datetime"), "M/d/yyyy")      
).drop("datetime")                            

pricing = wheat_price_data.withColumn(
    "Date",
    to_date(col("Date"), "M/d/yyyy")        
)

#Join on Date
merged = weather.join(pricing, on="Date", how="left")

merged.show(10, truncate=False)

#strip column names
old_cols = merged.columns
new_cols = [c.replace('.', '').replace(' ', '_') for c in old_cols]
merged_clean = merged.toDF(*new_cols)

#check for nulls
null_exprs = []
for c, dtype in merged_clean.dtypes:
    cond = col(c).isNull() | (isnan(col(c)) if dtype in ("double","float") else col(c).isNull())
    null_exprs.append(count(when(cond, col(c))).alias(f"{c}_missing"))

merged_clean.select(*null_exprs).show(truncate=False)

#Summary statistics
merged_clean.describe().show(truncate=False)

+----------+-----+-------+-------+----+------------+------------+---------+----+--------+------+----------+-----------+----------+----+---------+--------+---------+-------+----------------+----------+----------+--------------+-----------+-------+----------+-------------------+-------------------+---------+----------------------------+--------------------------------------------------------------------------------+-----------------+----------------------------------------------------------+------+------+------+------+------+--------+
|Date      |name |tempmax|tempmin|temp|feelslikemax|feelslikemin|feelslike|dew |humidity|precip|precipprob|precipcover|preciptype|snow|snowdepth|windgust|windspeed|winddir|sealevelpressure|cloudcover|visibility|solarradiation|solarenergy|uvindex|severerisk|sunrise            |sunset             |moonphase|conditions                  |description                                                                     |icon             |stations                  

pyspark.sql.dataframe.DataFrame